**Notebook: Tools (Module 6)**
- OCBC AI Creator Masterclass, Day 2

___

This notebook is the hands-on part of Module 6 (about 38 minutes). After it come the example script
`day2/examples/m6_example.py` and the exercise `day2/exercises/m6_exercise.py`. Run it from the top, and run each
LLM cell **once** (quota).

# Background

Day 1 in one line: *we tell the LLM what the data is.* Day 2: **the LLM asks for the data it needs.**

The RMs now want to ask anything: *"How did NISP close?"*, *"Which bank has the best ROE?"*, *"Why did BBRI fall?"*
You cannot write a script for every question. Instead, you give the LLM **tools** (your Day 1 functions with a
description) and let it ask for the one a question needs:

- **6.1 From function to tool.** What the LLM sees of a plain Python function.
- **6.2 How tool calling works.** The LLM asks; our code runs. One round trip by hand.
- **6.3 The description decides.** Why the docstring is the most important line.
- **6.4 The news search becomes a tool.** Module 5's retriever as `search_news`.
- **6.5 Choosing the tools.** Few tools, read apart from action.

| Step on the ladder | Who decides WHICH data | Who decides WHEN |
|---|---|---|
| Report in the prompt (M3–M4) | Our code | The scheduler |
| RAG (M5) | The user's question | The user |
| **Tool calling (M6)** | **The LLM, one choice** | **The user asks, the LLM picks** |
| Agent (M7) | The LLM, step by step | The LLM |

## Training Objectives

- Turn a Sectors function into a tool (a plain function with a docstring), and read what the LLM sees with `Tool(f).tool_def`
- Run one tool call by hand: the LLM's request, our code running it, the result back to the LLM
- Write descriptions and arguments that make the LLM choose the right tool with the right arguments
- Turn the Module 5 retriever into a `search_news` tool
- Choose the few tools an agent needs, and tell read tools from action tools

## Checking your answers

**Your turn** cells say what the right result is. Run the next cell once for the **Knowledge Check** questions.

In [ ]:
import hashlib

ANSWERS = {'m6_sees': '5e2edef6c5', 'm6_runs': '1322c19180', 'm6_empty': '5aa3879d5c', 'm6_when_search': 'ede76d8949', 'm6_choose': 'e2b49fd491', 'm6_action': '64bb2d9594'}

def check_answer(question, answer):
    key = hashlib.sha256(f"{question}:{answer.strip().lower()}".encode()).hexdigest()[:10]
    if answer.strip() == "":
        print("Type a, b, c or d between the quotes, then run the cell again.")
    elif key == ANSWERS[question]:
        print("Correct.")
    else:
        print("Not quite. Read the question again and try another answer.")

In [ ]:
import os
from dotenv import load_dotenv

# Work from the project root (the folder with data/), so file paths are the same as in the .py scripts
while not os.path.exists("data/sample"):
    os.chdir("..")

load_dotenv()  # reads the keys from .env
print("Working folder:", os.getcwd())

# 6.1 From Function to Tool

> **The request**
>
> An RM: **"How did NISP close?"** You already have the function. Now the LLM must be able to use it.

> **In this section**
>
> - **Key question:** What does the LLM see of a tool?
> - **How we run it:** **Live, about 8 minutes.**
> - **What you will do:** write two Sectors functions as tools, read their name, description and arguments, then write a tool yourself.
> - **What carries forward:** `get_company_overview` and `get_daily_prices`, used in every section below.

In Pydantic AI a tool is a **plain function**. It reads three things from it: the function **name**, the **docstring**
(the description the LLM reads; the lines under `Args:`, after a blank line, describe each argument) and the
**arguments with their types**. The LLM never sees the code inside. `Tool(f).tool_def` shows what the LLM will see.
A tool is still a function: our code calls it as usual, `get_company_overview("NISP")`.

In [ ]:
import requests
from datetime import date, timedelta
from pydantic_ai import Tool

BASE_URL = "https://api.sectors.app/v2"
HEADERS = {"Authorization": os.getenv("SECTORS_API_KEY")}


def get_company_overview(ticker: str) -> dict:
    """Get the profile and latest closing price of one company on the Indonesia Stock Exchange.
    Use it for questions about a single company, e.g. "How did NISP close?".

    Args:
        ticker: 4-letter IDX code, e.g. BBCA, BBRI, BMRI, NISP.
    """
    response = requests.get(f"{BASE_URL}/company/report/{ticker}/", headers=HEADERS,
                            params={"sections": "overview"}, timeout=30)
    return response.json()


def get_daily_prices(ticker: str, days: int = 10) -> list:
    """Get the daily closing prices and volumes of one IDX company for the last `days` calendar days.
    Use it for questions about how a stock moved recently, e.g. "How did BMRI do this week?".

    Args:
        ticker: 4-letter IDX code, e.g. BMRI.
        days: how many calendar days back to look (a week is 7).
    """
    end = date.today()
    response = requests.get(f"{BASE_URL}/daily/{ticker}/", headers=HEADERS,
                            params={"start": str(end - timedelta(days=days)), "end": str(end)}, timeout=30)
    return response.json()


tool_def = Tool(get_company_overview).tool_def
print("Name:", tool_def.name)
print("Description:", tool_def.description)
print("Arguments:", tool_def.parameters_json_schema["properties"])
print(get_company_overview("NISP")["overview"]["last_close_price"])

#### Your turn: A ranking tool

Write the tool `get_top_banks(metric: str = "market_cap", n: int = 5) -> dict` that ranks the banks with the
Sectors screener (Module 2: `/companies/`, `where` on `sub_sector='banks'`, `order_by` on `-{metric}`, `limit`,
`include_query_values`). The docstring must say what it returns, when to use it, and list the metrics the LLM may use:
`market_cap`, `roe_ttm` (return on equity), `pe_ttm`, `pb_mrq`, `yield_ttm`. Call it for `roe_ttm` with `n=3` and
print the symbols. The right result is 3 bank symbols, and `Tool(get_top_banks).tool_def.parameters_json_schema`
shows `metric` and `n`, each with its description.

In [ ]:
## Your code below



#### Knowledge Check: What the LLM sees

Which part of a tool does the LLM **not** see?

- **a.** The name
- **b.** The docstring
- **c.** The argument names and types
- **d.** The code inside the function

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m6_sees", answer)

# 6.2 How Tool Calling Works

> **The request**
>
> Your manager: **"Before we trust it, show me exactly who does what when the LLM 'uses' a tool."**

> **In this section**
>
> - **Key question:** Who runs the tool: the LLM or our code?
> - **How we run it:** **Live, about 10 minutes.**
> - **What you will do:** send the tool descriptions with a question, read the tool call, run it, send the result back, and get the answer; then do a round trip yourself.
> - **What carries forward:** the round trip that a Pydantic AI `Agent` runs for us in Module 7.

1. We send the question **and the tool descriptions** (`ModelRequestParameters(function_tools=...)`).
2. The LLM answers with a **tool call**: a name and arguments. It runs nothing.
3. **Our code** runs the tool and sends the result back as a `ToolReturnPart` with the call's `tool_call_id`.
4. The LLM writes the answer from the result.

`model_request` (from `pydantic_ai.direct`) sends one request to the model, with no agent around it, so we can see
each step. In a notebook we `await` it; in a .py script the same call is `model_request_sync(...)`.

In [ ]:
from pydantic_ai.direct import model_request
from pydantic_ai.messages import ModelRequest, ToolReturnPart
from pydantic_ai.models import ModelRequestParameters
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIResponsesModel(
    os.getenv("OPENAI_MODEL", "gpt-6.1-sol"),
    provider=OpenAIProvider(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_URL")),
)

tools = [Tool(get_company_overview), Tool(get_daily_prices)]
tools_by_name = {t.name: t for t in tools}
params = ModelRequestParameters(function_tools=[t.tool_def for t in tools])   # the tool descriptions

messages = [ModelRequest.user_text_prompt("How did NISP close on its last trading day?")]
response = await model_request(model, messages, model_request_parameters=params)   # step 2: the LLM asks
print("Text:", repr(response.text))
print("Tool calls:", response.tool_calls)

In [ ]:
messages.append(response)
returns = []
for call in response.tool_calls:                            # step 3: OUR code runs the tool
    result = tools_by_name[call.tool_name].function(**call.args_as_dict())
    returns.append(ToolReturnPart(tool_name=call.tool_name, content=result, tool_call_id=call.tool_call_id))
messages.append(ModelRequest(parts=returns))

final = await model_request(model, messages, model_request_parameters=params)   # step 4: the LLM answers from the result
print(final.text)

#### Your turn: A round trip of your own

Do the four steps yourself for *"How did BMRI shares do over the past week?"*: a new message list, the LLM's tool
call, your code running it, the `ToolReturnPart`, the final answer. Print the tool call and the answer. The right result
is a call to `get_daily_prices` with `ticker` `BMRI` and `days` about 7, and an answer that quotes BMRI's closes.

In [ ]:
## Your code below



#### Knowledge Check: Who runs it?

When the LLM "uses" `get_daily_prices`, where does the request to Sectors come from?

- **a.** From the model provider's servers, run by the LLM
- **b.** From our program, which runs the tool the LLM asked for
- **c.** From Sectors, which reads the prompt
- **d.** From Pydantic AI's cloud

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m6_runs", answer)

# 6.3 The Description Decides

> **The request**
>
> An RM: **"Which three banks have the highest return on equity?"** Will the LLM pick your ranking tool?

> **In this section**
>
> - **Key question:** How does the LLM choose a tool, and its arguments?
> - **How we run it:** **Live, about 7 minutes.**
> - **What you will do:** offer the same function with a vague and a clear description, then test your own `get_top_banks`.
> - **What carries forward:** the docstring habits for every tool in Modules 7–9.

In [ ]:
def tool_a(ticker: str) -> list:
    """Gets data."""
    return get_daily_prices(ticker, days=7)


def tool_b(ticker: str) -> list:
    """Daily closing prices of one IDX stock for the last 7 days.
    Use it for questions like "How did BMRI do this week?".

    Args:
        ticker: 4-letter IDX code, e.g. BMRI.
    """
    return get_daily_prices(ticker, days=7)


params_ab = ModelRequestParameters(function_tools=[Tool(tool_a).tool_def, Tool(tool_b).tool_def])
question = "How did BMRI shares do over the past week?"
choice = await model_request(model, [ModelRequest.user_text_prompt(question)], model_request_parameters=params_ab)
print("The LLM picked:", [c.tool_name for c in choice.tool_calls])

#### Your turn: Test your description

Send `get_company_overview`, `get_daily_prices` and **your** `get_top_banks` as tools and ask *"Which three banks have
the highest return on equity?"* Print the tool calls. The right result is `get_top_banks` with `metric` `roe_ttm` and
`n` 3. If the LLM picks another tool or another metric, improve your docstring and try again: that is the skill.

In [ ]:
## Your code below



#### Knowledge Check: No tool call

`response.tool_calls` is empty and the answer contains a price that looks made up. The most likely cause?

- **a.** The tools were not sent with the question, or no description matches the question
- **b.** The Sectors key is wrong
- **c.** The tool returned too much data
- **d.** The internet is slow

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m6_empty", answer)

# 6.4 The News Search Becomes a Tool

> **The request**
>
> An RM: **"Why did BBRI fall so sharply in mid October?"** Prices say *that* it fell; the news says *why*.

> **In this section**
>
> - **Key question:** How does a retriever become a tool, and how can the LLM narrow the search?
> - **How we run it:** **Live, about 8 minutes.**
> - **What you will do:** rebuild the Module 5 vector store, wrap the search in `search_news`, then give it a ticker argument.
> - **What carries forward:** `search_news`, which the agent uses in Modules 7–9.

In Module 5 every question searched the news. As a tool, the LLM decides **when** to search, and writes the search
words itself. The vector store is rebuilt from the saved news (the same `data/sample/` copy and the same open-source
embedding model as in Module 5), with the Module 5 functions copied in:

In [ ]:
import json
import numpy as np
from pydantic_ai import Embedder

embedder = Embedder("sentence-transformers:all-mpnet-base-v2")   # open source, runs on your laptop


def split_text(text, chunk_size=1500, chunk_overlap=100):
    """Cut a text into pieces of chunk_size characters; each piece repeats the last chunk_overlap characters of the one before."""
    chunks, start = [], 0
    while True:
        chunks.append(text[start:start + chunk_size])
        if start + chunk_size >= len(text):
            return chunks
        start += chunk_size - chunk_overlap


def split_documents(docs, chunk_size=1500, chunk_overlap=100):
    return [{"text": piece, "metadata": doc["metadata"]}
            for doc in docs for piece in split_text(doc["text"], chunk_size, chunk_overlap)]


def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


async def build_vector_store(chunks):
    vectors = (await embedder.embed_documents([c["text"] for c in chunks])).embeddings
    return {"chunks": chunks, "vectors": np.array(vectors)}


async def similarity_search(vector_store, query, k=4, filter=None):
    query_vector = (await embedder.embed_query(query))[0]
    scored = [(cosine_similarity(query_vector, v), c) for v, c in zip(vector_store["vectors"], vector_store["chunks"])]
    scored = [(s, c) for s, c in scored if filter is None or filter(c)]
    return [c for s, c in sorted(scored, key=lambda pair: pair[0], reverse=True)[:k]]


docs, seen = [], set()
for ticker in ["BBCA", "BBRI", "BMRI", "NISP"]:
    with open(f"data/sample/news_{ticker}.json", encoding="utf-8") as f:
        for article in json.load(f):
            if article["title"] not in seen:
                seen.add(article["title"])
                docs.append({"text": article["title"] + "\n\n" + article["body"],
                             "metadata": {"title": article["title"], "date": article["timestamp"][:10], "ticker": ticker}})

chunks = split_documents(docs)
vector_store = await build_vector_store(chunks)


async def search_news(query: str) -> str:
    """Search recent Indonesian market news. Use it for every question that asks WHY something happened
    (events, announcements, reasons): the reason is in the news, not in the prices. It does not give prices.

    Args:
        query: what to look for, in plain words.
    """
    results = await similarity_search(vector_store, query, k=4)
    return "\n\n".join(f"{c['metadata']['title']} ({c['metadata']['date']})\n{c['text']}" for c in results)


print(len(chunks), "chunks")
params_news = ModelRequestParameters(
    function_tools=[Tool(f).tool_def for f in [get_company_overview, get_daily_prices, search_news]])
question = "Why did BBRI shares fall so sharply in mid-October 2026?"
response = await model_request(model, [ModelRequest.user_text_prompt(question)], model_request_parameters=params_news)
print(response.tool_calls)

#### Your turn: Let the LLM narrow the search

Write `search_bank_news(query: str, ticker: str = "")`: the same search, but when `ticker` is given it keeps only
chunks of that bank (the metadata filter from Module 5). Say in the docstring when to fill `ticker`. Send it with
`get_daily_prices` and ask *"Why did BBRI shares fall so sharply in mid-October 2026?"* The right result is a call to
`search_bank_news` with `ticker` `BBRI` and a query about the fall (often together with a `get_daily_prices` call).

In [ ]:
## Your code below



#### Knowledge Check: When to search

With both the price tools and `search_news` offered, which question should lead to `search_news`?

- **a.** How did NISP close yesterday?
- **b.** Which bank has the highest ROE?
- **c.** What did OCBC NISP announce about acquisitions?
- **d.** What was BMRI's volume on Friday?

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m6_when_search", answer)

# 6.5 Choosing the Tools

> **The request**
>
> Your manager: **"Can we give the agent all 30 functions we have, just in case?"**

> **In this section**
>
> - **Key question:** How many tools, and which kind?
> - **How we run it:** **Discussion, about 5 minutes.**
> - **What you will do:** decide which tools a job needs and which need a person's approval.
> - **What carries forward:** the tool list of the Module 7 agent.

Every tool description is sent with **every** call: more tools means a longer prompt and more chances of the wrong
choice. Give an agent the **3 to 5 tools** its job needs. Keep **read** tools (they fetch data) apart from **action**
tools (they change something: save a file, send an email); action tools need a person's approval (Module 7). And when
your code already knows which data is needed, as for the 07:30 report, call the function directly: no tool needed.

#### Knowledge Check: Tools for a chatbot

A chatbot answers *"Which banks are the largest?"*, *"How did NISP trade this week?"* and *"Why did BBRI fall?"*.
Which tools should it get?

- **a.** All the tools we have
- **b.** `get_top_banks`, `get_daily_prices`, `search_news`
- **c.** `get_company_overview` only
- **d.** `search_news` only: the news mentions prices

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m6_choose", answer)

#### Knowledge Check: Read or action?

Which tool needs a person's approval before it runs?

- **a.** `get_top_banks`
- **b.** `search_news`
- **c.** `send_email_to_client`
- **d.** `get_daily_prices`

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m6_action", answer)

# Module 6 Wrap-up

| Idea | In code |
|---|---|
| A tool = function + description | plain function, docstring, typed arguments; `Tool(f).tool_def` |
| The LLM only asks | `ModelRequestParameters(function_tools=...)` → `response.tool_calls` |
| Our code runs it | `tool.function(**call.args_as_dict())` → `ToolReturnPart(..., tool_call_id=call.tool_call_id)` |
| The description decides | when to use it, what it returns, allowed values |
| Few tools, read vs action | 3–5 tools; actions need approval |

**Next:** `day2/examples/m6_example.py`, then the exercise `day2/exercises/m6_exercise.py`: two new Sectors tools and
the round trip as a function. In Module 7 a Pydantic AI `Agent` runs this loop for us, as many rounds as needed.